In [1]:
import logging
log_level = logging.INFO
# log_level = logging.DEBUG
log_level = 25
logging.basicConfig(level=log_level)
logger = logging.getLogger(__name__)
logger.setLevel(log_level)


In [2]:
import os
import pandas as pd
from openai import OpenAI
from regulations_rag.rerank import RerankAlgos
from gdpr_rag.gdpr_chat import GDPRChat


In [3]:
openai_client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"),)
key = os.getenv('encryption_key_gdpr')

rerank_algo  = RerankAlgos.MOST_COMMON
#rerank_algo  = RerankAlgos.LLM
chat = GDPRChat(openai_client = openai_client, 
                    decryption_key = key, 
                    rerank_algo = rerank_algo, 
                    user_name_for_logging = 'test_user')

#chat.rerank_algo  = RerankAlgos.MOST_COMMON


In [9]:
user_content = "Under what conditions could you be exempt from keeping records of your data processing?"
chat.user_provides_input(user_content)

ANALYSIS:regulations_rag.regulation_chat:user to test_user: Under what conditions could you be exempt from keeping records of your data processing?
ANALYSIS:regulations_rag.regulation_chat:assistant to test_user: You could be exempt from keeping records of your data processing activities if your enterprise or organization employs fewer than 250 persons and if the processing you carry out does not result in a risk to the rights and freedoms of data subjects, is occasional, or does not include special categories of data as referred to in Article 9(1) or personal data relating to criminal convictions and offences referred to in Article 10.  
Reference:  
30: Chapter IV Controller and processor. Section 1 General obligations. Article 30 Records of processing activities.


In [6]:
print(chat.reader.get_regulation_detail("4"))

Article 4 Definitions
    1. 'personal data' means any information relating to an identified or identifiable natural person ('data subject'); an identifiable natural person is one who can be identified, directly or indirectly, in particular by reference to an identifier such as a name, an identification number, location data, an online identifier or to one or more factors specific to the physical, physiological, genetic, mental, economic, cultural or social identity of that natural person;
    2. 'processing' means any operation or set of operations which is performed on personal data or on sets of personal data, whether or not by automated means, such as collection, recording, organisation, structuring, storage, adaptation or alteration, retrieval, consultation, use, disclosure by transmission, dissemination or otherwise making available, alignment or combination, restriction, erasure or destruction;
    3. 'restriction of processing' means the marking of stored personal data with the

In [8]:
df = chat.index.index
for index, row in df[df["section_reference"].str.startswith("30")].iterrows():
    print(f"*   {row['text']}")

*   What records must you keep about your data processing activities?
*   What details do records of data processing need to include?
*   Are there any specific requirements for the records of data processing to be kept by a data processor?
*   In what form should records of data processing be maintained?
*   Who might request access to records of data processing, and what is your responsibility?
*   Under what conditions could you be exempt from keeping records of your data processing?
*   You must maintain a record of all processing activities for which you are responsible. This record must include:
- Your name and contact details, and if relevant, those of any joint controllers, your representative, and your data protection officer.
- The purposes for processing the data.
- Descriptions of the individuals and types of personal data you are processing.
- Categories of recipients to whom the data has been or will be disclosed, including any in third countries or international organisa

In [11]:
print(chat._create_system_message())

You are answering questions for a Controller based only on the sections from the General Data Protection Regulation (GDPR) that are provided. Please use the manual's index pattern when referring to sections: (\d{1,2})(?:\((\d{1,2})\))?(?:\(([a-z])\))?. You have three options:
1) Answer the question. Preface an answer with the tag 'ANSWER:'. End the answer with 'Reference: ' and a comma separated list of the section you used to answer the question if you used any.
2) Request additional documentation. If, in the body of the sections provided, there is a reference to another section of the Manual that is directly relevant and not already provided, respond with the word 'SECTION:' followed by the full section reference.
3) State 'NONE:' and nothing else in all other cases

"
